# Unit 3 — Multimedia and Interactive Visualization (UE24CS342AA9)
## Activity Notebook: The Parking-Lot Incident Review

### Scenario

You are the new visual-analytics intern at **Northgate Tech Park Security**. Last night the camera over
Parking Lot B recorded something, and the Head of Security wants a briefing **today**. You have the 2-minute
camera log (`unit3_ppt1_cctv_log.csv`), the camera still, and 10 s of microphone audio. Their message to you:

1. *"Before anyone asks for 'just store everything in full HD', tell me how much data our 400 cameras would
   produce per day. I need real numbers."*
2. *"Our night camera images always look murky. Can you prove what's wrong with them and fix one?"*
3. *"I am not watching two minutes of video. Give me a timeline with the moments that matter and one
   picture per moment."*
4. *"Something banged around 75 s. Was it a **gunshot** or **breaking glass**? Those trigger completely
   different responses."*
5. *"Put it all on ONE screen for the briefing."*

Cells marked **`# TODO`** are for you to complete. Markdown cells marked **Reflection** are for you to answer
in your own words.

## Part 0 — Setup (given)

Run this cell as-is. It loads the camera log, the camera still, a frame renderer, and the audio (plus two
**reference recordings** from the security vendor's sound library: a known gunshot and a known glass break).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cbook
from scipy import ndimage, signal

np.random.seed(0)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

FPS, FS = 30, 16_000
cctv = pd.read_csv("unit3_ppt1_cctv_log.csv")
img = plt.imread(cbook.get_sample_data("grace_hopper.jpg")).astype(float)
lum = 0.299 * img[..., 0] + 0.587 * img[..., 1] + 0.114 * img[..., 2]
night_still = 18 + 0.22 * lum + np.random.normal(0, 1.5, lum.shape)        # what the night camera delivers

FRAME_H, FRAME_W = 90, 160
def render_frame(row):
    yy, xx = np.mgrid[0:FRAME_H, 0:FRAME_W]
    base = row.mean_brightness * (0.75 + 0.25 * yy / FRAME_H)
    frame = np.dstack([base, base, base * 1.05]); frame[60:62, :, :] += 25
    if not np.isnan(row.car_x):
        frame[(np.abs(xx - row.car_x / 4) < 14) & (np.abs(yy - row.car_y / 4) < 6)] = [200, 40, 40]
    if not np.isnan(row.person_x):
        frame[((xx - row.person_x / 4) / 2.2) ** 2 + ((yy - row.person_y / 4) / 5) ** 2 <= 1] = [30, 90, 230]
    return np.clip(frame, 0, 255).astype(np.uint8)

def _clip(kind, seed):
    rng = np.random.default_rng(seed)
    t = np.arange(int(2.0 * FS)) / FS
    x = 0.004 * rng.normal(size=t.size)
    m = t >= 0.5
    if kind == "gunshot":                       # sharp broadband crack + low boom, nothing after
        x[m] += 0.9 * np.exp(-(t[m] - 0.5) / 0.02) * rng.normal(size=m.sum())
        x[m] += 0.5 * np.exp(-(t[m] - 0.5) / 0.25) * np.sin(2 * np.pi * 70 * (t[m] - 0.5))
    else:                                       # impact + lingering high-frequency tinkle
        x[m] += 0.9 * np.exp(-(t[m] - 0.5) / 0.03) * rng.normal(size=m.sum())
        for _ in range(40):
            t0 = 0.55 + rng.exponential(0.35); f0 = rng.uniform(3000, 7000)
            mk = (t >= t0) & (t < t0 + 0.06)
            x[mk] += 0.12 * np.exp(-(t[mk] - t0) / 0.015) * np.sin(2 * np.pi * f0 * (t[mk] - t0))
    return t, x

ref_gunshot = _clip("gunshot", 11)
ref_glass = _clip("glass", 12)
rng = np.random.default_rng(0)
t_audio = 70 + np.arange(10 * FS) / FS
x_audio = 0.02 * np.sin(2 * np.pi * 120 * t_audio) + 0.004 * rng.normal(size=t_audio.size)
_, unk = _clip("glass", 99)
x_audio[(t_audio >= 74.5) & (t_audio < 76.5)] += unk[: int(2 * FS)]           # the mystery sound

print(f"Camera log: {len(cctv):,} frames | still: {night_still.shape} | audio: {x_audio.size:,} samples at {FS} Hz")
cctv.head()

## Task 1 — The Storage Reality Check (Request #1)

**Real-world usage:** Storage budgets for camera fleets are decided from exactly these back-of-envelope numbers.

**Why this technique:** Uncompressed storage = **W × H × C × B** per frame (Slides 25–28), frames = **f × T**
(Slide 35), audio samples = **f_s × T** (Slide 55).

**TODO:**
1. Compute the uncompressed size (in MB) of ONE 1920 × 1080 RGB frame at 8 bits per channel.
2. Compute the number of frames ONE camera records in 24 h at 30 fps.
3. Compute the total uncompressed video for 400 cameras for 24 h, in **terabytes** (1 TB = 1024⁴ bytes).
4. Compute the same total if the cameras were downsampled to 640 × 360 and 10 fps. What percentage of the
   pixels per frame remain after the downsample?

In [ ]:
# TODO 1: one 1080p RGB frame, in MB
frame_bytes = 1920 * 1080 * 3 * (8 / 8)          # W x H x C x B(bytes)
frame_mb = frame_bytes / 1024**2

# TODO 2: frames per camera per day at 30 fps
SECONDS_PER_DAY = 24 * 3600
frames_per_day = 30 * SECONDS_PER_DAY

# TODO 3: 400 cameras x 24 h, uncompressed, in TB
fleet_tb = frame_bytes * frames_per_day * 400 / 1024**4

# TODO 4: same fleet at 640x360 and 10 fps (TB), and % of pixels per frame that remain
small_frame_bytes = 640 * 360 * 3
fleet_tb_small = small_frame_bytes * (10 * SECONDS_PER_DAY) * 400 / 1024**4
pct_pixels_remaining = 100 * (640 * 360) / (1920 * 1080)

print(f"One frame: {frame_mb:.2f} MB | frames/day: {frames_per_day:,}")
print(f"Fleet (1080p, 30 fps): {fleet_tb:,.1f} TB/day | Fleet (360p, 10 fps): {fleet_tb_small:,.1f} TB/day")
print(f"Pixels per frame remaining after downsample: {pct_pixels_remaining:.2f}%")

**Reflection:** Downsampling to 360p/10 fps saves a huge amount of storage. Using Slides 13 and 36–37
(raster interpolation; frame rate and the correspondence problem), name **one spatial** and **one temporal**
piece of evidence the security team could lose forever by making that choice.

_Your answer:_

## Task 2 — Diagnose and Fix the Murky Night Image (Request #2)

**Real-world usage:** Before anyone interprets a night image, an analyst checks its histogram to see whether
the problem is exposure, contrast or clipping.

**Why this technique:** The **histogram** shows the intensity distribution (Slides 16–18); **Michelson
contrast** C = (L_max − L_min)/(L_max + L_min) quantifies it (Slide 15); a **contrast stretch** remaps the
occupied range onto 0–255.

**TODO:**
1. Plot `night_still` and its histogram (64 bins, range 0–255) side by side.
2. Compute its Michelson contrast using the 1st and 99th percentiles as L_min and L_max (robust to outliers).
3. Contrast-stretch it: map the 1st percentile → 0 and the 99th percentile → 255 (clip to 0–255).
4. Plot the stretched image + histogram, and print the new contrast.

In [ ]:
def michelson(lmax, lmin):
    return (lmax - lmin) / (lmax + lmin)

# TODO 2: robust Michelson contrast of night_still
lo, hi = np.percentile(night_still, [1, 99])
c_before = michelson(hi, lo)

# TODO 3: contrast-stretched version of night_still (1st pct -> 0, 99th pct -> 255)
stretched = np.clip((night_still - lo) / (hi - lo) * 255, 0, 255)

lo2, hi2 = np.percentile(stretched, [1, 99])
c_after = michelson(hi2, lo2)

fig, axes = plt.subplots(2, 2, figsize=(10, 8), gridspec_kw={"height_ratios": [2, 1]})
# TODO 1 & 4: top row = images, bottom row = histograms (before | after)
for col, (data, name) in enumerate([(night_still, "Night image (original)"),
                                    (stretched, "After contrast stretch")]):
    axes[0, col].imshow(data, cmap="gray", vmin=0, vmax=255)
    axes[0, col].set_title(name)
    axes[0, col].axis("off")
    axes[1, col].hist(data.ravel(), bins=64, range=(0, 255), color="slategray")
    axes[1, col].set_xlabel("Intensity (0-255)")
    axes[1, col].set_ylabel("Pixels")

plt.tight_layout(); plt.show()
print(f"Contrast before: {c_before:.3f}  ->  after: {c_after:.3f}")

**Reflection:** Describe the *shape* of the night image's histogram in the lecture's vocabulary (Slide 18 table).
Then explain why stretching made the image readable even though it added **no new information** — and why a
histogram alone could never have told you *what* is in the picture (Slide 18, "Same Histogram ≠ Same Image").

_Your answer:_

## Task 3 — From Two Minutes of Video to a One-Glance Timeline (Request #3)

**Real-world usage:** Security reviewers search for events; they don't watch footage linearly.

**Why this technique:** **Motion vectors → events → timeline + representative frames** (Slides 38–40, 45, 47).
Content-based keyframes beat uniform sampling when activity is sparse.

**TODO:**
1. Compute the person's speed in px/s as the magnitude of their motion vector over a 1-second window
   (`diff(FPS)` on x and y — single-frame differences are dominated by tracking jitter).
2. Find the time ranges where the person is **standing** (speed < 3 px/s) and **running** (speed > 25 px/s).
3. Detect the frames where `audio_level_db` first rises above −20 dB and where `mean_brightness` jumps by more
   than 40 in one frame.
4. Plot a timeline of `motion_energy` with a vertical marker + label for each event you found.
5. Show **one keyframe per event** (use `render_frame(cctv.iloc[i])`).

In [ ]:
t = cctv["time_s"].to_numpy()
px_, py_ = cctv["person_x"], cctv["person_y"]

# TODO 1: person speed (px/s): displacement over a 1-second window (FPS frames)
speed = np.hypot(px_.diff(FPS), py_.diff(FPS)).to_numpy()   # NaN wherever the person is missing

# TODO 2: standing / running time ranges (print the first and last time of each)
def runs(mask, min_seconds=1.0, bridge_seconds=1.0):
    """Frame ranges where mask is True; bridge short gaps, drop very short runs."""
    mask = np.asarray(mask, dtype=bool)
    out = []
    i = 0
    while i < len(mask):
        if mask[i]:
            j = i
            while j + 1 < len(mask) and mask[j + 1]:
                j += 1
            if out and (i - out[-1][1]) / FPS <= bridge_seconds:
                out[-1][1] = j
            else:
                out.append([i, j])
            i = j + 1
        else:
            i += 1
    return [(a, b) for a, b in out if (b - a) / FPS >= min_seconds]

valid = ~np.isnan(speed)
standing = runs(valid & (np.nan_to_num(speed, nan=999) < 3))
running = runs(valid & (np.nan_to_num(speed, nan=0) > 25))
for name, rr in [("Standing", standing), ("Running", running)]:
    for a, b in rr:
        print(f"{name}: {t[a]:.1f} s -> {t[b]:.1f} s")

# TODO 3: event frame indices
audio_event_idx = int(np.argmax(cctv["audio_level_db"].to_numpy() > -20))
lighting_event_idx = int(np.argmax(cctv["mean_brightness"].diff().to_numpy() > 40))
print(f"Audio event at {t[audio_event_idx]:.2f} s | lighting event at {t[lighting_event_idx]:.2f} s")

car_in = np.flatnonzero(cctv["car_x"].notna().to_numpy())
person_in = np.flatnonzero(px_.notna().to_numpy())
longest_stand = max(standing, key=lambda r: r[1] - r[0])
events = {
    "Car enters": int(car_in[0]),
    "Person appears": int(person_in[0]),
    "Person stands still": longest_stand[0],
    "Loud bang": audio_event_idx,
    "Person runs": running[0][0],
    "Person leaves": int(person_in[-1]),
    "Lights on": lighting_event_idx,
}
events = dict(sorted(events.items(), key=lambda kv: kv[1]))

# TODO 4: annotated motion-energy timeline
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(t, cctv["motion_energy"], lw=1, color="tab:blue")
ymax = cctv["motion_energy"].max()
for n, (label, idx) in enumerate(events.items()):
    ax.axvline(t[idx], ls="--", lw=0.8, color="tab:red")
    ax.text(t[idx] + 0.4, ymax * (0.97 - 0.1 * n), f"{label} ({t[idx]:.1f} s)", fontsize=7, color="tab:red", va="top")
ax.set(xlabel="Time (s)", ylabel="Motion energy", title="Motion energy with detected events")
plt.tight_layout(); plt.show()

# TODO 5: one keyframe per event
fig, axes = plt.subplots(2, 4, figsize=(14, 5.5))
for ax in axes.ravel():
    ax.axis("off")
for ax, (label, idx) in zip(axes.ravel(), events.items()):
    ax.imshow(render_frame(cctv.iloc[idx]))
    ax.set_title(f"{label}\n{t[idx]:.1f} s", fontsize=9)
plt.tight_layout(); plt.show()

**Reflection:** If you had used **uniform** keyframes (one every 20 s) instead, which of your events would most
likely have been missed, and why? Which lecture concept (Slide 40) names the approach you used instead?

_Your answer:_

## Task 4 — Gunshot or Glass? (Request #4)

**Real-world usage:** Acoustic event classification in security systems starts with an analyst comparing
spectrogram signatures against known references.

**Why this technique:** A **waveform** shows *when* something loud happened but not *what* it was; a
**spectrogram** (Slides 56–58) reveals each sound's time–frequency signature.

**TODO:**
1. Plot the waveforms of `ref_gunshot`, `ref_glass` and the mystery segment of `x_audio` (74.5–76.5 s).
2. Plot their spectrograms (use `signal.spectrogram(x, fs=FS, nperseg=512, noverlap=384)`, show in dB,
   y-axis 0–8000 Hz).
3. Compute, for each clip, the fraction of spectral energy **above 3 kHz** in the 0.2–1.2 s after the impact.
4. Print a one-line verdict.

In [ ]:
mystery = x_audio[(t_audio >= 74.5) & (t_audio < 76.5)]
clips = {"Reference: gunshot": ref_gunshot[1], "Reference: glass break": ref_glass[1], "MYSTERY (74.5-76.5 s)": mystery}

fig, axes = plt.subplots(2, 3, figsize=(15, 6))
# TODO 1 & 2: waveform (top row) and spectrogram (bottom row) for each clip
time_axis = np.arange(int(2.0 * FS)) / FS
for col, (name, sig) in enumerate(clips.items()):
    axes[0, col].plot(time_axis, sig, lw=0.5, color="black")
    axes[0, col].set(title=name, xlabel="Time (s)", ylabel="Amplitude", ylim=(-1.2, 1.2))

    f, tt, S = signal.spectrogram(sig, fs=FS, nperseg=512, noverlap=384)
    axes[1, col].pcolormesh(tt, f, 10 * np.log10(S + 1e-12), shading="auto", cmap="magma", vmin=-110, vmax=-20)
    axes[1, col].set(ylim=(0, 8000), xlabel="Time (s)", ylabel="Frequency (Hz)")

plt.tight_layout(); plt.show()

# TODO 3: fraction of energy above 3 kHz, 0.2-1.2 s after the impact (impact is at 0.5 s in every clip)
hf_fraction = {}
for name, sig in clips.items():
    f, tt, S = signal.spectrogram(sig, fs=FS, nperseg=512, noverlap=384)
    after = S[:, (tt >= 0.7) & (tt <= 1.7)]
    hf_fraction[name] = float(after[f > 3000].sum() / after.sum())

print(hf_fraction)
# TODO 4: verdict
mystery_hf = hf_fraction["MYSTERY (74.5-76.5 s)"]
match = min(["Reference: gunshot", "Reference: glass break"], key=lambda k: abs(hf_fraction[k] - mystery_hf))
print(f"Verdict: {mystery_hf:.1%} of the mystery sound's energy is above 3 kHz -> matches the {match.split(': ')[1]}.")

**Reflection:** Look only at the three **waveforms**. Could you have confidently told the gunshot from the glass
break? What exactly does the spectrogram add (name the three dimensions from Slide 57), and which visual pattern
(horizontal band / vertical stripe / scattered high-frequency specks) decided your verdict?

_Your answer:_

## Task 5 — The One-Screen Briefing (Request #5)

**Real-world usage:** Incident briefings use a synchronized multimodal view so every claim is backed by two
modalities at the same timestamp.

**Why this technique:** **Image–video–audio integration** linked through a common time reference
(Slide 59), with **overview first** (timeline) and detail at the selected moment.

**TODO:** Build ONE figure with:
1. the rendered video frame at t = 75 s (left),
2. the full 120 s motion + audio timeline with a red marker at 75 s (top right),
3. the spectrogram of `x_audio` (70–80 s) with the same marker (bottom right),
4. a conclusion-style figure title (what happened, not "Incident Dashboard").

In [ ]:
t_sel = 75.0
fig = plt.figure(figsize=(14, 6))
gs = fig.add_gridspec(2, 2, width_ratios=[1, 2], height_ratios=[1, 1.3])
# TODO 1-4
t_all = cctv["time_s"].to_numpy()
i_sel = int(np.argmin(np.abs(t_all - t_sel)))

# 1) frame at t = 75 s
ax_f = fig.add_subplot(gs[:, 0])
ax_f.imshow(render_frame(cctv.iloc[i_sel]))
ax_f.set_title(f"Camera frame at {t_sel:.0f} s")
ax_f.axis("off")

# 2) motion energy + audio level over the whole log
ax_m = fig.add_subplot(gs[0, 1])
ax_m.plot(t_all, cctv["motion_energy"], color="tab:blue", lw=1)
ax_m.set(xlim=(0, 120), xlabel="Time (s)", ylabel="Motion energy", title="Whole 120 s log")
ax_a = ax_m.twinx()
ax_a.spines["right"].set_visible(True)
ax_a.plot(t_all, cctv["audio_level_db"], color="tab:orange", lw=0.8)
ax_a.set_ylabel("Audio level (dB)")
ax_m.axvline(t_sel, color="red", lw=2)

# 3) spectrogram of the 10 s of microphone audio
ax_s = fig.add_subplot(gs[1, 1])
f, tt, S = signal.spectrogram(x_audio, fs=FS, nperseg=512, noverlap=384)
ax_s.pcolormesh(70 + tt, f, 10 * np.log10(S + 1e-12), shading="auto", cmap="magma", vmin=-110, vmax=-20)
ax_s.set(ylim=(0, 8000), xlabel="Time (s)", ylabel="Frequency (Hz)", title="Microphone spectrogram, 70-80 s")
ax_s.axvline(t_sel, color="red", lw=2)

# 4) conclusion-style title
fig.suptitle("Glass broke at 75 s in Lot B; the person standing nearby ran off about 3.6 s later",
             fontsize=13, fontweight="bold")

plt.tight_layout(); plt.show()

**Reflection:** Slide 61 warns that combining modalities can cause **information overload** and depends on correct
**synchronization**. What would go wrong in your briefing if the audio track were offset by +2 s from the video?
Which panel would make the mistake visible?

_Your answer:_

## Task 6 — The Decision (No Code)

**Reflection (final):** Write a 4–6 sentence briefing note to the Head of Security covering the whole notebook.
It should:
- State what happened in Lot B in one sentence, with timestamps.
- Justify the **one** visualization you'd put first (timeline, keyframes, spectrogram or the synchronized view),
  using at least one lecture idea (e.g. structure → pattern → change → event → meaning, content-based keyframes,
  time × frequency × magnitude).
- Recommend a storage policy from Task 1 and name the one piece of evidence it must *not* sacrifice.

_Your final briefing here:_